# Building an Agent Loop from First Principles

This is my take on the exercise at the end of `5_extra.ipynb`: build an agent loop from scratch, with

1. A terminal UI (using `rich`) so the agent's steps are visible as they happen
2. A **checklist** tool, so the model plans its work as several tool calls and tracks progress

The loop has three parts:

- **Tools**: plain Python functions, plus a JSON schema for each one so the model knows how to call it
- **Dispatch**: when the model asks for a tool, run the matching function and send the result back
- **Loop**: keep calling the model until it stops asking for tools and gives a final answer

## 1. Setup

In [ ]:
import json
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
openai = OpenAI()
console = Console()

MODEL = "gpt-5.5"
MAX_TURNS = 15  # safety limit so a confused model can't loop forever

## 2. State: the checklist

The checklist is just a list of dictionaries kept in memory. Each item has its text and whether it is done.

In [ ]:
checklist = []

def checklist_report() -> str:
    """Plain-text version of the checklist, which is what the model sees."""
    if not checklist:
        return "The checklist is empty."
    lines = []
    for i, item in enumerate(checklist, start=1):
        status = "done" if item["done"] else "todo"
        lines.append(f"{i}. [{status}] {item['text']}")
    return "\n".join(lines)

def render_checklist():
    """Terminal view of the checklist, which is what the user sees."""
    table = Table(title="Checklist")
    table.add_column("#", justify="right")
    table.add_column("Task")
    table.add_column("Status")
    for i, item in enumerate(checklist, start=1):
        if item["done"]:
            table.add_row(str(i), f"[strike]{item['text']}[/strike]", "[green]done[/green]")
        else:
            table.add_row(str(i), item["text"], "[yellow]todo[/yellow]")
    console.print(table)

## 3. Tools

Each tool is an ordinary Python function. It returns a string, which goes back to the model as the tool result.

In [ ]:
def create_checklist(descriptions: list[str]) -> str:
    for description in descriptions:
        checklist.append({"text": description, "done": False})
    render_checklist()
    return checklist_report()

def mark_complete(index: int, completion_notes: str) -> str:
    if not 1 <= index <= len(checklist):
        return f"Error: there is no checklist item {index}."
    console.print(f"[dim]Step {index} notes:[/dim] {completion_notes}")
    checklist[index - 1]["done"] = True
    render_checklist()
    return checklist_report()

def list_checklist() -> str:
    render_checklist()
    return checklist_report()

The model can't see these Python functions. It only sees the **schemas** below: a name, a description, and the parameters. The descriptions matter because the model uses them to decide when to call each tool.

In [ ]:
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "create_checklist",
            "description": "Add one or more steps to the checklist. Returns the full checklist.",
            "parameters": {
                "type": "object",
                "properties": {
                    "descriptions": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "The steps to add, in order",
                    }
                },
                "required": ["descriptions"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "mark_complete",
            "description": "Mark one checklist step as done. Returns the full checklist.",
            "parameters": {
                "type": "object",
                "properties": {
                    "index": {
                        "type": "integer",
                        "description": "The 1-based number of the step to mark as done",
                    },
                    "completion_notes": {
                        "type": "string",
                        "description": "How the step was completed, in rich console markup",
                    },
                },
                "required": ["index", "completion_notes"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "list_checklist",
            "description": "Return the current checklist without changing it. Useful for checking progress.",
            "parameters": {
                "type": "object",
                "properties": {},
                "additionalProperties": False,
            },
        },
    },
]

# Map tool names to functions explicitly, rather than looking them up in globals()
TOOL_FUNCTIONS = {
    "create_checklist": create_checklist,
    "mark_complete": mark_complete,
    "list_checklist": list_checklist,
}

## 4. Dispatch

When the model asks for a tool, we look up the function by name, call it with the arguments the model supplied, and wrap the result in a `tool` message. The message must carry the same `tool_call_id` as the request, so the model knows which result belongs to which call.

Errors are caught and sent back as text. That way a bad call (for example, an unknown tool, or markup that `rich` can't parse) becomes information the model can act on, instead of crashing the loop.

In [ ]:
def run_tool_call(tool_call) -> dict:
    name = tool_call.function.name
    try:
        arguments = json.loads(tool_call.function.arguments)
        function = TOOL_FUNCTIONS.get(name)
        if function is None:
            result = f"Error: unknown tool '{name}'."
        else:
            result = function(**arguments)
    except Exception as e:
        result = f"Error while running {name}: {e}"
    return {"role": "tool", "tool_call_id": tool_call.id, "content": str(result)}

## 5. The agent loop

Each pass through the loop:

1. Send the whole conversation so far to the model
2. If the reply has no tool calls, it is the final answer: show it and stop
3. Otherwise, record the model's tool-call message, run each tool, add the results, and go round again

The conversation list is the agent's memory. Every tool call and result is appended to it, so the model sees what it has already done.

In [ ]:
SYSTEM_MESSAGE = """
You are given a problem. First use the checklist tools to write a short plan of steps.
Then carry out each step in order, and mark each one complete once you have done it.
If a quantity is missing from the problem, add a step to choose a reasonable estimate.
When every step is done, reply with the solution in Rich console markup, without code blocks.
Do not ask the user questions. Answer only after using your tools.
"""

def run_agent(user_message: str, system_message: str = SYSTEM_MESSAGE, max_turns: int = MAX_TURNS):
    checklist.clear()
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]

    for turn in range(1, max_turns + 1):
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=TOOL_SCHEMAS)
        message = response.choices[0].message

        # No tool calls means the model has finished and this is its answer
        if not message.tool_calls:
            console.print(Panel(message.content or "", title="Answer", border_style="green"))
            return message.content

        console.print(f"[dim]Turn {turn}: the model asked for {len(message.tool_calls)} tool call(s)[/dim]")

        # Record the model's request so the tool results have something to answer
        messages.append({
            "role": "assistant",
            "content": message.content,
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {"name": tc.function.name, "arguments": tc.function.arguments},
                }
                for tc in message.tool_calls
            ],
        })
        for tool_call in message.tool_calls:
            messages.append(run_tool_call(tool_call))

    console.print(f"[red]Stopped after {max_turns} turns without a final answer.[/red]")
    return None

## 6. Check the tools without calling the model

Before involving the API, I test the checklist tools directly. This also shows what the TUI looks like.

In [ ]:
create_checklist(["Buy milk", "Write report"])
mark_complete(1, "Bought two cartons from the corner shop.")
print(mark_complete(9, "This index does not exist."))
print(list_checklist())
checklist.clear()

## 7. Run the agent

The same train problem from `5_extra.ipynb`. The question leaves out the distance between Boston and New York, so the agent has to add a step to estimate it.

In [ ]:
USER_MESSAGE = """
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""

answer = run_agent(USER_MESSAGE)

## 8. Things to try

### 8a. A different problem: a shopping list with a budget

Here the agent has to plan several purchases and keep track of the total. Nothing is given about the prices, so it has to estimate them.

In [ ]:
budget_message = """
I have $40 to spend on groceries for a week of breakfasts and lunches for one person.
Plan a shopping list that fits the budget, and tell me what it costs in total and what I have left over.
"""

budget_answer = run_agent(budget_message)

### 8b. The safety limit: `MAX_TURNS = 2`

With only two turns, the agent does not have time to finish. The loop should stop and print a message instead of running forever.

In [ ]:
limited_answer = run_agent(USER_MESSAGE, max_turns=2)
print("Returned value:", limited_answer)

### 8c. The new `list_checklist` tool

The tool is registered, so I call it directly first. Then I ask the agent to check its own progress, so it can use the tool during a run.

In [ ]:
checklist.clear()  # start from an empty list, since section 7 left items behind
create_checklist(["Write the outline", "Rehearse the talk"])
list_checklist()
checklist.clear()

progress_message = """
Make a checklist of three steps for planning a 10-minute presentation.
Mark the first step complete, then use the list_checklist tool to check progress before you reply.
Reply with a one-line summary of the plan.
"""

progress_answer = run_agent(progress_message)